# Chapter 05 Companion Notebook: Model Evaluation: Classification

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch05_Model_Evaluation_Classification.ipynb)

This notebook accompanies Chapter 05 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).



# Classification Model Evaluation

### Use “Customer churn.csv”
- Churn: 1 if the customer has left the company or discontinued their service
- AccountWeeks: the number of weeks a customer has been with the company
- DataPlan: 1 if the customer is subscribed to a data plan
- DataUsage: the amount of data a customer has consumed
- CustServCalls: the number of times a customer has contacted customer service
- DayMins: the total number of minutes a customer has used during the day
- DayCalls: the number of calls a customer has made during the day
- MonthlyCharge: the amount a customer is billed each month
- OverageFee: charges a customer incurs for exceeding their plan's limits
- RoamMins: the number of minutes a customer spends on roaming calls

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn import metrics
import matplotlib.pyplot as plt
import seaborn as sns

### 1. Report the average values of the following variables by Churn vs. No-churn.
- AccountWeeks, DataPlan, DataUsage, CustServCalls, DayMins, DayCalls, MonthlyCharge, OverageFee, RoamMins

In [ ]:
# Read data

df= pd.read_csv('Customer churn.csv')
df.head()

- `df = pd.read_csv()` reads data from a CSV file named "Telco churn.csv" and loads it into a pandas DataFrame.

In [ ]:
# Calculate average by 'churn'

df.groupby('Churn').mean()

- `df.groupby('Churn').mean()` groups the data by the 'Churn' column and calculates the mean of the numerical columns for each group.

### 2. Divide the data into 75% training and 25% test set (use random_state=10) and run the following logistic regression model on training data. Report coefficients including intercept.
- Dependent variable: Churn
- Independent variables: AccountWeeks, DataPlan, DataUsage, CustServCalls, DayMins, DayCalls, MonthlyCharge, OverageFee, RoamMins

In [ ]:
# Define y, x

y=df.Churn
x=df[['AccountWeeks', 'DataPlan', 'DataUsage', 'CustServCalls', 'DayMins', 'DayCalls', 
      'MonthlyCharge', 'OverageFee', 'RoamMins']]

- `x = df[[...]]`: x is a DataFrame that includes a set of independent variables or features.

In [ ]:
# Divide x into training and test sets. Run logistic regression and obtain coefficients.

xtrain, xtest, ytrain, ytest=train_test_split(x, y, random_state=10)
m = LogisticRegression(random_state=10, max_iter=1000).fit(xtrain, ytrain)
m.coef_

- `train_test_split(x, y, random_state=10)` splits the data into training and testing sets.
- `m = LogisticRegression(random_state=10).fit(xtrain, ytrain)` creates a logistic regression model (m) and fits it to the training data.
- `m.coef_` retrieves the coefficients of the logistic regression model.

### 4. Report model accuracy, confusion matrix, precision, recall, and F1 score on test data.

In [ ]:
# Predict y and obtain scores

pred = m.predict(xtest)
m.score(xtest, ytest)  # accuracy
print('Accuracy', metrics.accuracy_score(ytest, pred))
print('Precision', metrics.precision_score(ytest, pred))  # TP/P*
print('Recall', metrics.recall_score(ytest, pred))  # TP/P
print('F1 Score', metrics.f1_score(ytest, pred))

- `pred = m.predict(xtest)`: Make predictions using the model m on the test data xtest, and the predictions are stored in the variable pred.
- `m.score(xtest, ytest)`: Calculates the accuracy of the model m on the test data. It computes the fraction of correctly classified instances. The xtest contains the input features, and ytest contains the true labels.
- `metrics.accuracy_score(ytest, pred)`: Calculates the accuracy of the model using accuracy_score function. It compares the true labels ytest with the predicted labels pred and prints the accuracy.
- `metrics.precision_score(ytest, pred)`: Calculates the precision of the model. Precision is a measure of how many of the positively predicted instances were actually positive.
- `metrics.recall_score(ytest, pred)`: Calculates the recall of the model. Recall is a measure of how many of the actual positive instances were correctly predicted as positive.
- `metrics.f1_score(ytest, pred)`: Calculates the F1 score of the model. The F1 score is the harmonic mean of precision and recall, providing a single metric that balances both.

In [ ]:
# Obtain classification report

print(classification_report(ytest, pred))

- `classification_report` function calculates several metrics, including precision, recall, F1-score, and support for each class.

In [ ]:
# Obtain confusion matrix

metrics.confusion_matrix(ytest, pred)

- `metrics.confusion_matrix(ytest, pred)` calculates the confusion matrix for a classification model's predictions. The confusion matrix is a table that summarizes the performance of a classification algorithm by counting the number of true positives (TP), true negatives (TN), false positives (FP), and false negatives (FN).

|             | Predicted 0 (P*) | Predicted 1 (N*) |
|-------------|:----------------:|-----------------:|
| Actual 0 (N)|        TN        |        FP        |
| Actual 1 (P)|        FN        |        TP        |

### 5. Draw ROC curve with AUC on test data.

In [ ]:
# Predict probabilities on test data

prob = m.predict_proba(xtest)
prob  # column 1,2: prob of negative and positive class

- `prob = m.predict_proba(xtest)` Used to obtain the predicted probabilities for each class for the instances in the test dataset.
    - `m` is trained classification model.
    - `xtest` is the test dataset containing input features for which you want to make predictions.
    - `predict_proba` returns a matrix or array where each row corresponds to an instance in the xtest dataset, and each column corresponds to a class label.

In [ ]:
# Obtain predictions for positive class, fpr, tpr, decisin threshold, AUC

prob = m.predict_proba(xtest)[:, 1]  # all rows, 2nd column
fpr, tpr, threshold = roc_curve(ytest, prob)
 # roc_curve takes observed labels and predicted prob, then returns fpr,tpr,thresholds for each tpr&fpr

roc_auc = auc(fpr, tpr)
print('Out-Sample AUC: %0.4f' % roc_auc)  # '%0.4f' %: display 4 digit after decimal

- `prob = m.predict_proba(xtest)[:, 1]`: Extracts the predicted probabilities for the positive class (class 1) from the predict_proba output.
- `fpr, tpr, threshold = roc_curve(ytest, prob)`: Calculates the False Positive Rate (FPR), True Positive Rate (TPR), and the threshold values for various cutoff points.
- `roc_auc = auc(fpr, tpr)`: Calculates the Area Under the ROC Curve (AUC). The AUC is a single scalar value that represents the overall performance of the binary classification model.
- `print('Out-Sample AUC: %0.4f' % roc_auc)`: This line prints the AUC with four decimal places.
    - '%0.4f' % displays the AUC value with four digits after the decimal point.

In [ ]:
# Disply ROC curve

plt.plot(fpr, tpr, label='AUC = %0.3f)' % roc_auc)
plt.plot([0, 1], [0, 1]) # straight line
plt.grid()
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc="lower right")
# plt.figure(figsize=(10,7)) # enlarge size
# plt.title('ROC Curve')

- `plt.plot(fpr, tpr, label='AUC = %0.3f)' % roc_auc)`: Creates a plot of the ROC curve using fpr (False Positive Rate) on the x-axis and tpr (True Positive Rate) on the y-axis.
- `plt.plot([0, 1], [0, 1])`: Adds a straight line to the plot. The straight line represents the ROC curve of a random classifier (no discrimination power), which serves as a reference line.
- `plt.grid()`: This line adds a grid to the plot, making it easier to read and interpret the graph.
- `plt.xlabel('False Positive Rate')`: This line labels the x-axis as "False Positive Rate."
- `plt.ylabel('True Positive Rate')`: This line labels the y-axis as "True Positive Rate."
- `plt.legend(loc="lower right")`: This line adds a legend to the plot, and it specifies that the legend should appear in the lower right corner of the plot.

### 6. What is the optimal decision threshold (probability) that maximizes (TPR-FPR)?

In [ ]:
# Obtain the optimal decision threshold

best_thresh = threshold[np.argmax(tpr-fpr)]
 # np.argmax returns the index of the maximum value
best_thresh

- `np.argmax(tpr - fpr)` calculates the index of the threshold where the difference between TPR and FPR is maximized.
- `threshold[np.argmax(tpr - fpr)]` retrieves the threshold value corresponding to the index found in the previous step.

In [ ]:
print(max(tpr-fpr))  # maximum of (TPR-FPR)
np.argmax(tpr-fpr)  # Index of maximum of (TPR-FPR)

### 7. Report model accuracy, confusion matrix, precision, recall, and F1 score on test data using the optimal threshold. Discuss differences in results based on the two thresholds (default and optimal).

In [ ]:
# Predict y using the optimal threshold and obtain scores

pred1 = (prob >= best_thresh).astype('int')  # convert Boolean to integer
print('Accuracy',  metrics.accuracy_score(ytest, pred1))
print('Precision', metrics.precision_score(ytest, pred1))
print('Recall',    metrics.recall_score(ytest, pred1))
print('F1 Score',  metrics.f1_score(ytest, pred1))

- `pred1 = (prob >= best_thresh).astype('int')` is creating binary class predictions based on the calculated threshold value best_thresh.
    - `(prob >= best_thresh)` creates a Boolean array that compares each element of prob to best_thresh. If the probability is greater than or equal to the threshold, it evaluates to True, indicating a positive prediction. If it's below the threshold, it evaluates to False, indicating a negative prediction.
    - `astype('int')` converts the Boolean values to integers, where True becomes 1 and False becomes 0.

In [ ]:
# Obtain confusion matrix

metrics.confusion_matrix(ytest, pred1)